# Materials Project Structure Discovery with CrossCat (v2)

**v2 additions:** Compositional features (electronegativity, ionic radius) + ORDINAL column type (Laue symmetry class). See v1 notebook for the original 20-column analysis.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sambhal-labs/jaxcross/blob/main/examples/materials_project_discovery_v2.ipynb)

Every ML paper on Materials Project data uses **supervised learning** to predict a single property
(band gap, bulk modulus, dielectric constant) from crystal structure. No one is doing
**unsupervised joint structure discovery** across all material properties simultaneously.

This notebook demonstrates CrossCat's unique capability: discovering which material properties
are jointly dependent, which are independent, and which materials are anomalous across the
full property landscape — **without any labels**.

**Dataset:** ~7,000 materials with dielectric data from [Materials Project](https://materialsproject.org/)
v2025.09.25, enriched with elasticity (~72% missing), piezoelectric, and summary properties.
The natural sparsity pattern is ideal for CrossCat's native NaN handling.

**Key outputs:**
- **Dependence structure (Z-matrix):** Which material properties co-vary?
- **Anomaly detection:** Which materials have unusual property combinations?
- **Missing property imputation:** Predict expensive-to-compute elasticity from cheaper electronic properties
- **Mutual information:** Quantify nonlinear property relationships
- **Generative classification:** Predict metallicity without a classifier

## 1. Setup — Install jaxcross and Materials Project Client

In [ ]:
!nvidia-smi

In [ ]:
import os

WORKDIR = "/kaggle/working/jaxcross"
BRANCH = "main"  # @param {type:"string"}

!git clone https://github.com/sambhal-labs/jaxcross.git {WORKDIR} 2>/dev/null \
    || (cd {WORKDIR} && git pull)
os.chdir(WORKDIR)

!git fetch origin && (git checkout {BRANCH} || git checkout -b {BRANCH} origin/{BRANCH}) && git pull origin {BRANCH}

# Preserve Kaggle's pre-installed JAX+CUDA stack to avoid ptxas version mismatch.
%pip install -e . --no-deps -q
%pip install mp-api seaborn matplotlib scikit-learn -q

print(f"Branch: {BRANCH}")
print(f"Working directory: {os.getcwd()}")

## 2. Verify GPU and Imports

In [ ]:
import json
import time
from collections import Counter
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from emmet.core.summary import HasProps
from IPython.display import Image, display
from mp_api.client import MPRester
from pymatgen.core import Element
from pymatgen.symmetry.groups import SpaceGroup

import crosscat
import crosscat.packed.state as _ps
from crosscat import (
    batch_classify_column,
    batch_impute_column,
    batch_packed_states,
    batch_row_typicality,
    estimate_packed_memory,
    initialize,
    load_latest_checkpoint,
    pack_state,
    packed_dependence_matrix,
    packed_mutual_information,
    packed_predictive_probability,
    save_checkpoint,
    save_packed_state,
    suggest_max_clusters,
    unpack_state,
)
from crosscat.diagnostics import effective_sample_size, gelman_rubin_rhat
from crosscat.packed import packed_gibbs_sweep
from crosscat.packed.kernels import packed_log_joint
from crosscat.types import ColumnType

# Platform info
devices = jax.devices()
n_devices = len(devices)
backend = jax.default_backend()
print(f"jaxcross {crosscat.__version__}")
print(f"JAX backend: {backend}, devices: {n_devices}")
for d in devices:
    print(f"  {d}")

## 3. Configuration

Adjust parameters based on your GPU. With 2xT4 (Kaggle), chains are distributed
across devices via `jax.pmap` for true multi-GPU parallelism.

| Parameter | 2xT4 (Kaggle) | P100 | A100 |
|-----------|---------------|------|------|
| N_CHAINS | 4 (2/device) | 4 | 8 |
| N_SWEEPS | 200 | 300 | 400 |
| MAX_CLUSTERS | 32 | 64 | 128 |


In [ ]:
# ---- Configuration ----
N_CHAINS = 8
N_SWEEPS = 1100
DIAG_EVERY = 100  # Check convergence every N sweeps
CKPT_EVERY = 100  # Checkpoint every N sweeps
MAX_VIEWS = 16
MAX_CLUSTERS = 32
SEED = 42
MIN_COVERAGE = 0.25  # Drop columns with < 25% non-null (elasticity is ~28%)

# Multi-GPU: distribute chains across devices
CHAINS_PER_DEVICE = max(1, N_CHAINS // n_devices)
N_CHAINS = CHAINS_PER_DEVICE * n_devices  # Round to divisible
print(f"Multi-GPU: {N_CHAINS} chains, {CHAINS_PER_DEVICE} per device, {n_devices} devices")

# Materials Project API key
# On Kaggle: store as a secret named "MP_API_KEY"
try:
    from kaggle_secrets import UserSecretsClient

    MP_API_KEY = UserSecretsClient().get_secret("MP_API_KEY")
    print("API key loaded from Kaggle secrets")
except ImportError:
    import os

    MP_API_KEY = os.environ.get("MP_API_KEY", "YOUR_API_KEY_HERE")
    print("API key loaded from environment")

CACHE_DIR = Path("examples/results/materials_project")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PATH = CACHE_DIR / "mp_dielectric_cache_v2.parquet"

print(f"Config: {N_CHAINS} chains x {N_SWEEPS} sweeps, seed={SEED}")

## 4. Data Loading — Materials Project Dielectric Dataset

We fetch all materials that have dielectric property data (~7K materials), then enrich with
elasticity and piezoelectric properties. The elasticity subset has ~72% sparsity among
the dielectric materials — this natural missingness pattern is exactly what makes this a
great CrossCat table.

**Column types:**
- CONTINUOUS: band gap, formation energy, density, dielectric constants, elastic moduli
- BINARY: is_stable, is_metal
- CATEGORICAL: crystal system (7 values), magnetic ordering

In [ ]:
# Column catalog: (attribute_name, display_name, ColumnType)
# Organized by property domain for interpretability.

COLUMN_CATALOG = [
    # === Electronic ===
    ("band_gap", "Band Gap (eV)", ColumnType.CONTINUOUS),
    ("is_metal", "Is Metal", ColumnType.BINARY),
    ("e_electronic", "Electronic Dielectric", ColumnType.CONTINUOUS),
    ("e_ionic", "Ionic Dielectric", ColumnType.CONTINUOUS),
    ("e_total", "Total Dielectric", ColumnType.CONTINUOUS),
    # Note: Refractive index (n) is sqrt(e_electronic) — a mathematical identity,
    # not an independent property. We omit it to avoid inflating dependence scores.
    # === Thermodynamic ===
    ("formation_energy_per_atom", "Formation Energy (eV/atom)", ColumnType.CONTINUOUS),
    ("energy_above_hull", "E Above Hull (eV/atom)", ColumnType.CONTINUOUS),
    ("is_stable", "Is Stable", ColumnType.BINARY),
    # === Structural / Compositional ===
    ("density", "Density (g/cm3)", ColumnType.CONTINUOUS),
    ("volume", "Volume (A3)", ColumnType.CONTINUOUS),
    ("nsites", "N Sites", ColumnType.CONTINUOUS),
    ("nelements", "N Elements", ColumnType.CONTINUOUS),
    ("crystal_system", "Crystal System", ColumnType.CATEGORICAL),
    # === Mechanical (sparse — ~72% missing in dielectric subset) ===
    ("bulk_modulus_vrh", "Bulk Modulus (GPa)", ColumnType.CONTINUOUS),
    ("shear_modulus_vrh", "Shear Modulus (GPa)", ColumnType.CONTINUOUS),
    ("universal_anisotropy", "Elastic Anisotropy", ColumnType.CONTINUOUS),
    ("homogeneous_poisson", "Poisson Ratio", ColumnType.CONTINUOUS),
    # === Piezoelectric (sparse — ~55% missing) ===
    ("e_ij_max", "Piezo e_ij_max", ColumnType.CONTINUOUS),
    # === Compositional (derived from pymatgen Composition) ===
    ("avg_electroneg", "Avg Electronegativity", ColumnType.CONTINUOUS),
    ("avg_ionic_radius", "Avg Ionic Radius (A)", ColumnType.CONTINUOUS),
    # === Symmetry — ORDINAL type (ordered logistic model) ===
    # Laue class: 11 tiers ordered by rotational symmetry (low -> high)
    ("laue_class", "Laue Class", ColumnType.ORDINAL),
    # === Magnetic ===
    ("total_magnetization", "Magnetization", ColumnType.CONTINUOUS),
    ("ordering", "Magnetic Ordering", ColumnType.CATEGORICAL),
]

attr_names = [c[0] for c in COLUMN_CATALOG]
display_names = [c[1] for c in COLUMN_CATALOG]
column_types = [c[2] for c in COLUMN_CATALOG]
n_cols_catalog = len(COLUMN_CATALOG)
print(f"Column catalog: {n_cols_catalog} columns")
for attr, disp, ct in COLUMN_CATALOG:
    print(f"  {disp:30s}  {ct.name}")

In [ ]:
# Laue class mapping: space group -> point group -> Laue class (0-10, ordered by symmetry)
LAUE_MAP = {
    '1': 0, '-1': 0,
    '2': 1, 'm': 1, '2/m': 1,
    '222': 2, 'mm2': 2, 'mmm': 2,
    '4': 3, '-4': 3, '4/m': 3,
    '422': 4, '4mm': 4, '-42m': 4, '-4m2': 4, '4/mmm': 4,
    '3': 5, '-3': 5,
    '32': 6, '3m': 6, '-3m': 6,
    '6': 7, '-6': 7, '6/m': 7,
    '622': 8, '6mm': 8, '-6m2': 8, '-62m': 8, '6/mmm': 8,
    '23': 9, 'm-3': 9,
    '432': 10, '-43m': 10, 'm-3m': 10,
}
LAUE_NAMES = ['-1', '2/m', 'mmm', '4/m', '4/mmm', '-3', '-3m', '6/m', '6/mmm', 'm-3', 'm-3m']

def sg_to_laue_int(sg_number):
    """Map space group number (1-230) to Laue class index (0-10)."""
    try:
        sg = SpaceGroup.from_int_number(int(sg_number))
        return LAUE_MAP.get(sg.point_group)
    except Exception:
        return None

# Fetch data from Materials Project API (with local Parquet cache)

if CACHE_PATH.exists():
    print(f"Loading cached data from {CACHE_PATH}")
    df = pd.read_parquet(CACHE_PATH)
    print(f"Loaded: {df.shape}")
else:
    print("Fetching materials with dielectric data from Materials Project...")
    print("(This may take 2-5 minutes on first run)")
    t0 = time.time()

    with MPRester(MP_API_KEY) as mpr:
        # Step 1: Fetch summary docs for all materials with dielectric data
        print("  Fetching summary docs (has_props=dielectric)...")
        summary_docs = mpr.materials.summary.search(
            has_props=[HasProps.dielectric],
        )
        print(f"  Got {len(summary_docs)} summary docs")

        # Step 2: Extract material IDs for enrichment queries
        mpids = [str(doc.material_id) for doc in summary_docs]

        # Step 3: Fetch dielectric tensors
        print("  Fetching dielectric data...")
        dielectric_docs = mpr.materials.dielectric.search(material_ids=mpids)
        print(f"  Got {len(dielectric_docs)} dielectric docs")

        # Step 4: Fetch elasticity (subset — many materials lack elastic data)
        print("  Fetching elasticity data...")
        elastic_docs = mpr.materials.elasticity.search(material_ids=mpids)
        print(f"  Got {len(elastic_docs)} elasticity docs")

        # Step 5: Fetch piezoelectric
        print("  Fetching piezoelectric data...")
        piezo_docs = mpr.materials.piezoelectric.search(material_ids=mpids)
        print(f"  Got {len(piezo_docs)} piezoelectric docs")

    # Build summary DataFrame
    summary_records = []
    for doc in summary_docs:
        rec = {"material_id": str(doc.material_id)}
        for field in [
            "band_gap",
            "formation_energy_per_atom",
            "energy_above_hull",
            "density",
            "volume",
            "nsites",
            "is_stable",
            "is_metal",
            "total_magnetization",
            "ordering",
            "nelements",
        ]:
            val = getattr(doc, field, None)
            rec[field] = val
        # Crystal system from symmetry
        if hasattr(doc, "symmetry") and doc.symmetry is not None:
            rec["crystal_system"] = str(getattr(doc.symmetry, "crystal_system", None))
        else:
            rec["crystal_system"] = None
        # --- v2: Compositional features ---
        if doc.composition is not None:
            rec["avg_electroneg"] = doc.composition.average_electroneg
            comp_dict = doc.composition.as_dict()
            total = sum(comp_dict.values())
            if total > 0:
                weighted_ir = sum(
                    float(Element(el).average_ionic_radius) * amt
                    for el, amt in comp_dict.items()
                ) / total
                rec["avg_ionic_radius"] = weighted_ir if weighted_ir > 0 else None
            else:
                rec["avg_ionic_radius"] = None
        else:
            rec["avg_electroneg"] = None
            rec["avg_ionic_radius"] = None

        # --- v2: Laue class (ORDINAL) from space group ---
        if hasattr(doc, "symmetry") and doc.symmetry is not None:
            sg_num = getattr(doc.symmetry, "number", None)
            rec["laue_class"] = sg_to_laue_int(sg_num) if sg_num else None
        else:
            rec["laue_class"] = None

        # Store formula for labeling
        rec["formula_pretty"] = str(getattr(doc, "formula_pretty", ""))
        summary_records.append(rec)
    df_summary = pd.DataFrame(summary_records)

    # Build dielectric DataFrame
    df_diel = pd.DataFrame(
        [
            {
                "material_id": str(d.material_id),
                "e_total": getattr(d, "e_total", None),
                "e_ionic": getattr(d, "e_ionic", None),
                "e_electronic": getattr(d, "e_electronic", None),
                "n": getattr(d, "n", None),
            }
            for d in dielectric_docs
        ]
    )

    # Build elasticity DataFrame
    elastic_records = []
    for d in elastic_docs:
        rec = {"material_id": str(d.material_id)}
        if hasattr(d, "bulk_modulus") and d.bulk_modulus is not None:
            rec["bulk_modulus_vrh"] = getattr(d.bulk_modulus, "vrh", None)
        if hasattr(d, "shear_modulus") and d.shear_modulus is not None:
            rec["shear_modulus_vrh"] = getattr(d.shear_modulus, "vrh", None)
        rec["universal_anisotropy"] = getattr(d, "universal_anisotropy", None)
        rec["homogeneous_poisson"] = getattr(d, "homogeneous_poisson", None)
        elastic_records.append(rec)
    df_elastic = pd.DataFrame(elastic_records)

    # Build piezoelectric DataFrame
    df_piezo = pd.DataFrame(
        [
            {
                "material_id": str(d.material_id),
                "e_ij_max": getattr(d, "e_ij_max", None),
            }
            for d in piezo_docs
        ]
    )

    # Merge all on material_id
    df = df_summary.merge(df_diel, on="material_id", how="left")
    df = df.merge(df_elastic, on="material_id", how="left")
    df = df.merge(df_piezo, on="material_id", how="left")

    elapsed = time.time() - t0
    print(f"\nFetched and merged in {elapsed:.1f}s")
    print(f"Final shape: {df.shape}")

    # Cache to Parquet
    df.to_parquet(CACHE_PATH, index=False)
    print(f"Cached to {CACHE_PATH}")

print(f"\nDataset: {df.shape[0]} materials x {df.shape[1]} columns")
df.head()

In [ ]:
# Clean and encode columns for CrossCat

# Encode crystal_system as 0-indexed integer
CRYSTAL_SYSTEM_MAP = {
    "Triclinic": 0,
    "Monoclinic": 1,
    "Orthorhombic": 2,
    "Tetragonal": 3,
    "Trigonal": 4,
    "Hexagonal": 5,
    "Cubic": 6,
}
df["crystal_system"] = df["crystal_system"].map(CRYSTAL_SYSTEM_MAP)

# Encode magnetic ordering as 0-indexed integer
ordering_vals = sorted(df["ordering"].dropna().unique(), key=str)
ORDERING_MAP = {v: i for i, v in enumerate(ordering_vals)}
df["ordering"] = df["ordering"].map(ORDERING_MAP)
print(f"Crystal system encoding: {CRYSTAL_SYSTEM_MAP}")
print(f"Magnetic ordering encoding: {ORDERING_MAP}")

# Encode booleans as 0/1 float
for col in ["is_stable", "is_metal"]:
    df[col] = df[col].astype(float)

# ---- Data cleaning ----
# Replace infinities with NaN across all numeric columns
for col in df.select_dtypes(include=[np.number]).columns:
    n_inf = np.isinf(df[col].dropna()).sum()
    if n_inf > 0:
        df[col] = df[col].replace([np.inf, -np.inf], np.nan)
        print(f"Removed {n_inf} infinities from {col}")

# Remove unphysical values:
# - Negative elastic moduli are unphysical
# - Extreme outliers (beyond 99.9th percentile x 10) are likely DFT errors
CLEAN_COLS = [
    "bulk_modulus_vrh",
    "shear_modulus_vrh",
    "universal_anisotropy",
    "e_ionic",
    "e_electronic",
    "e_total",
]
# Note: Poisson ratio is NOT cleaned — negative values (auxetic materials) are physically valid
for col in CLEAN_COLS:
    if col not in df.columns:
        continue
    before = df[col].notna().sum()

    # Remove negative moduli
    if col in ["bulk_modulus_vrh", "shear_modulus_vrh"]:
        df.loc[df[col] <= 0, col] = np.nan

    # Remove extreme outliers (beyond 99.5th percentile x 5)
    s = df[col].dropna()
    if len(s) > 100:
        q995 = s.quantile(0.995)
        q005 = s.quantile(0.005)
        if q995 > 0:
            df.loc[df[col] > q995 * 5, col] = np.nan
        if q005 < 0:
            df.loc[df[col] < q005 * 5, col] = np.nan

    after = df[col].notna().sum()
    if before != after:
        print(f"Cleaned {col}: removed {before - after} invalid/outlier values")

# Log-transform highly skewed positive continuous columns
LOG_COLS = []
for attr in [
    "e_total",
    "e_ionic",
    "e_electronic",
    "volume",
    "nsites",
    "bulk_modulus_vrh",
    "shear_modulus_vrh",
    "e_ij_max",
]:
    if attr in df.columns:
        s = df[attr].dropna()
        if len(s) > 0 and s.min() > 0 and (s.max() / s.min()) > 100:
            df[attr] = np.log1p(df[attr])
            LOG_COLS.append(attr)

if LOG_COLS:
    print(f"Log-transformed (log1p): {LOG_COLS}")

# v2: Ionic radius zero-guard (some elements have 0 ionic radius)
if "avg_ionic_radius" in df.columns:
    df.loc[df["avg_ionic_radius"] == 0, "avg_ionic_radius"] = np.nan

# v2: Laue class is already 0-indexed integer (0-10), no encoding needed
if "laue_class" in df.columns:
    df["laue_class"] = df["laue_class"].astype(float)  # NaN-safe float encoding

# Summary of NaN pattern
print("\nColumn coverage (fraction non-null):")
for attr, disp, _ in COLUMN_CATALOG:
    if attr in df.columns:
        coverage = df[attr].notna().mean()
        print(f"  {disp:30s}  {coverage:.1%}")

In [ ]:
# Filter columns by coverage and visualize missingness

# Select only catalog columns that exist and meet coverage threshold
valid_attrs = []
valid_display = []
valid_types = []
for attr, disp, ct in COLUMN_CATALOG:
    if attr in df.columns:
        cov = df[attr].notna().mean()
        if cov >= MIN_COVERAGE:
            valid_attrs.append(attr)
            valid_display.append(disp)
            valid_types.append(ct)
        else:
            print(f"Dropping {disp}: coverage {cov:.1%} < {MIN_COVERAGE:.0%}")

df_model = df[valid_attrs].copy()
col_names = valid_display
column_types = valid_types
n_cols = len(col_names)
print(f"\nSelected {n_cols} columns for modeling")

# Missingness heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap (sample rows for readability)
sample_idx = np.random.default_rng(SEED).choice(
    len(df_model), size=min(200, len(df_model)), replace=False
)
sample_idx.sort()
miss_data = df_model.iloc[sample_idx].isna().values.astype(float)
axes[0].imshow(miss_data, aspect="auto", cmap="gray_r", interpolation="none")
axes[0].set_xlabel("Column")
axes[0].set_ylabel("Material (sampled)")
axes[0].set_title("Missingness Pattern (black = missing)")
axes[0].set_xticks(range(n_cols))
axes[0].set_xticklabels(col_names, rotation=45, ha="right", fontsize=7)

# Coverage bar chart
coverage_vals = [df_model[attr].notna().mean() for attr in valid_attrs]
bars = axes[1].barh(range(n_cols), coverage_vals, color="steelblue")
axes[1].set_yticks(range(n_cols))
axes[1].set_yticklabels(col_names, fontsize=8)
axes[1].set_xlabel("Coverage (fraction non-null)")
axes[1].set_title("Per-Column Data Coverage")
axes[1].axvline(
    x=MIN_COVERAGE, color="red", linestyle="--", label=f"Threshold ({MIN_COVERAGE:.0%})"
)
axes[1].legend()

plt.tight_layout()
plt.savefig(CACHE_DIR / "missingness.png", dpi=150, bbox_inches="tight")
plt.show()
display(Image(str(CACHE_DIR / "missingness.png")))
print(f"Overall missingness: {df_model.isna().mean().mean():.1%}")

In [ ]:
# Data summary
print(f"Dataset: {len(df_model)} materials x {n_cols} columns")
print("\nColumn type distribution:")

type_counts = Counter(ct.name for ct in column_types)
for ct_name, count in type_counts.most_common():
    print(f"  {ct_name}: {count}")

# Crystal system distribution
cs_col = "crystal_system" if "crystal_system" in valid_attrs else None
if cs_col:
    inv_cs_map = {v: k for k, v in CRYSTAL_SYSTEM_MAP.items()}
    cs_counts = df_model[cs_col].dropna().map(lambda x: inv_cs_map.get(int(x), "?")).value_counts()
    fig, ax = plt.subplots(figsize=(8, 4))
    cs_counts.plot.bar(ax=ax, color="steelblue")
    ax.set_title("Crystal System Distribution")
    ax.set_ylabel("Count")
    plt.tight_layout()
    plt.savefig(CACHE_DIR / "crystal_systems.png", dpi=150, bbox_inches="tight")
    plt.show()
    display(Image(str(CACHE_DIR / "crystal_systems.png")))

## 5. Preprocessing — Build JAX Arrays and Column Types

In [ ]:
# Extract metadata for labeling
material_ids = df["material_id"].tolist()
formulas = df["formula_pretty"].tolist()

# Convert to JAX array with NaN for missing
data_np = df_model.values.astype(np.float32)
data_jax = jnp.array(data_np)

n_rows, n_cols = data_jax.shape
nan_frac = float(jnp.isnan(data_jax).mean())

# Resource estimation
max_clusters = suggest_max_clusters(n_rows)
MAX_CLUSTERS = min(MAX_CLUSTERS, max_clusters)

print(f"Data array: {n_rows} rows x {n_cols} cols (float32)")
print(f"NaN fraction: {nan_frac:.1%}")
print(f"Max clusters: {MAX_CLUSTERS}")
print("\nColumn assignments:")
for name, ct in zip(col_names, column_types):
    print(f"  {name:30s}  {ct.name}")

## 6. Multi-Chain Gibbs Inference with pmap (Multi-GPU)

Collapsed Gibbs sampling discovers views (column groups) and clusters (row groups)
simultaneously. We use `jax.pmap` to distribute chains across GPUs — on Kaggle's
2xT4 setup, each GPU runs 2 chains in parallel for true multi-device acceleration.


In [ ]:
# Checkpoint and results directories (v2: use separate checkpoint dir)
CKPT_DIR = CACHE_DIR / "checkpoints_v2"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

import gc


# --- Define pmap sweep function ---
def _sweep_one_chain(key, packed, data, n_sweeps):
    """Run n_sweeps of packed Gibbs on a single chain."""
    return packed_gibbs_sweep(key, packed, data, n_sweeps=n_sweeps)


def _sweep_chains_on_device(keys, packed_batch, data, n_sweeps):
    """Run sweeps for CHAINS_PER_DEVICE chains on one device."""

    def body(i, carry):
        packed_b = carry
        single_kwargs = {}
        for name in _ps._ARRAY_FIELDS:
            single_kwargs[name] = getattr(packed_b, name)[i]
        for name in _ps._STATIC_FIELDS:
            single_kwargs[name] = getattr(packed_b, name)
        single = _ps.PackedCrossCatState(**single_kwargs)
        result = _sweep_one_chain(keys[i], single, data, n_sweeps)
        new_kwargs = {}
        for name in _ps._ARRAY_FIELDS:
            arr = getattr(packed_b, name)
            new_kwargs[name] = arr.at[i].set(getattr(result, name))
        for name in _ps._STATIC_FIELDS:
            new_kwargs[name] = getattr(packed_b, name)
        return _ps.PackedCrossCatState(**new_kwargs)

    return jax.lax.fori_loop(0, keys.shape[0], body, packed_batch)


pmap_sweep = jax.pmap(
    _sweep_chains_on_device,
    in_axes=(0, 0, None, None),
    static_broadcasted_argnums=(3,),
)


def _prepare_pmap(all_packed, chain_keys):
    """Batch chains and reshape for pmap distribution."""
    batched = batch_packed_states(all_packed)
    keys_pmap = chain_keys.reshape(n_devices, CHAINS_PER_DEVICE, *chain_keys.shape[1:])
    kwargs = {}
    for name in _ps._ARRAY_FIELDS:
        arr = getattr(batched, name)
        kwargs[name] = arr.reshape((n_devices, CHAINS_PER_DEVICE) + arr.shape[1:])
    for name in _ps._STATIC_FIELDS:
        kwargs[name] = getattr(batched, name)
    return keys_pmap, _ps.PackedCrossCatState(**kwargs)


def _unflatten_pmap(result_pmap, n_chains):
    """Extract individual packed states from pmap result."""
    out = []
    for c in range(n_chains):
        dev_idx = c // CHAINS_PER_DEVICE
        chain_idx = c % CHAINS_PER_DEVICE
        kwargs = {}
        for name in _ps._ARRAY_FIELDS:
            kwargs[name] = getattr(result_pmap, name)[dev_idx][chain_idx]
        for name in _ps._STATIC_FIELDS:
            kwargs[name] = getattr(result_pmap, name)
        out.append(_ps.PackedCrossCatState(**kwargs))
    return out


# --- Try to resume from checkpoint, otherwise initialize fresh ---
start_sweep = 0
all_packed = []
all_sweep_keys = []

try:
    loaded_packed, loaded_col_types, last_sweep = load_latest_checkpoint(str(CKPT_DIR))
    start_sweep = last_sweep
    print(f"Resumed from checkpoint: sweep {start_sweep}")
    # Use checkpoint as best chain, initialize remaining chains fresh
    key = jax.random.key(SEED)
    init_keys = jax.random.split(key, N_CHAINS)
    for chain_idx in range(N_CHAINS):
        k_i, k_sweep = jax.random.split(init_keys[chain_idx])
        all_sweep_keys.append(k_sweep)
        if chain_idx == 0:
            # First chain gets the checkpoint
            all_packed.append(loaded_packed)
            print(f"Chain {chain_idx + 1}: loaded from checkpoint (sweep {start_sweep})")
        else:
            result = initialize(k_i, data_jax, column_types)
            state = result.state
            all_packed.append(
                pack_state(state, max_views=MAX_VIEWS, max_clusters=MAX_CLUSTERS, data=data_jax)
            )
            print(f"Chain {chain_idx + 1}: initialized fresh ({state.n_views} views)")
            del state
    # Advance sweep keys past checkpoint to avoid key reuse
    for _ in range(start_sweep // DIAG_EVERY):
        for c in range(N_CHAINS):
            all_sweep_keys[c], _ = jax.random.split(all_sweep_keys[c])
except (FileNotFoundError, Exception) as e:
    print(f"No checkpoint found ({e}), initializing all chains fresh...")
    key = jax.random.key(SEED)
    init_keys = jax.random.split(key, N_CHAINS)
    for chain_idx in range(N_CHAINS):
        k_i, k_sweep = jax.random.split(init_keys[chain_idx])
        all_sweep_keys.append(k_sweep)
        result = initialize(k_i, data_jax, column_types)
        state = result.state
        all_packed.append(
            pack_state(state, max_views=MAX_VIEWS, max_clusters=MAX_CLUSTERS, data=data_jax)
        )
        print(f"Chain {chain_idx + 1}: initialized ({state.n_views} views)")
        del state

gc.collect()

mem_est = estimate_packed_memory(n_rows, n_cols, max_clusters=MAX_CLUSTERS, max_views=MAX_VIEWS)
total_bytes = sum(v for v in mem_est.values())
print(f"\nMemory per chain: {total_bytes / 1e6:.1f} MB")
print(f"Total memory ({N_CHAINS} chains): {total_bytes * N_CHAINS / 1e6:.1f} MB")

# --- pmap Gibbs sweep loop ---
log_joint_traces = [[] for _ in range(N_CHAINS)]
remaining_sweeps = N_SWEEPS - start_sweep

if remaining_sweeps <= 0:
    print(f"\nAlready completed {start_sweep} sweeps — skipping inference")
    # Score the existing chains
    scores = jnp.array([float(packed_log_joint(p, data_jax)) for p in all_packed])
    for c in range(N_CHAINS):
        log_joint_traces[c].append(float(scores[c]))
    total_time = 0.0
else:
    print(
        f"\nRunning sweeps {start_sweep + 1} to {N_SWEEPS} on {N_CHAINS} chains "
        f"({CHAINS_PER_DEVICE}/device, {n_devices} devices)..."
    )
    t_total = time.time()
    sweep = start_sweep

    while sweep < N_SWEEPS:
        batch = min(DIAG_EVERY, N_SWEEPS - sweep)

        # Split keys for this batch
        chain_keys_list = []
        for c in range(N_CHAINS):
            all_sweep_keys[c], subkey = jax.random.split(all_sweep_keys[c])
            chain_keys_list.append(subkey)
        chain_keys = jnp.stack(chain_keys_list)

        # pmap sweep across devices
        keys_pmap, batched_pmap = _prepare_pmap(all_packed, chain_keys)
        t0 = time.time()
        result_pmap = pmap_sweep(keys_pmap, batched_pmap, data_jax, batch)
        jax.tree.map(lambda x: x.block_until_ready(), result_pmap)
        sweep_time = time.time() - t0
        sweep += batch

        # Unflatten results
        all_packed = _unflatten_pmap(result_pmap, N_CHAINS)

        # Score each chain
        scores = jnp.array([float(packed_log_joint(p, data_jax)) for p in all_packed])
        for c in range(N_CHAINS):
            log_joint_traces[c].append(float(scores[c]))

        elapsed = time.time() - t_total

        # Convergence diagnostics
        if len(log_joint_traces[0]) >= 4:
            traces = jnp.array(log_joint_traces)
            rhat = float(gelman_rubin_rhat(traces))
            ess = float(effective_sample_size(traces))
            print(
                f"Sweep {sweep:4d} ({sweep_time:5.1f}s, total {elapsed:6.1f}s): "
                f"Rhat={rhat:.3f} ESS={ess:.1f} "
                f"log_joints=[{', '.join(f'{float(s):.0f}' for s in scores)}]"
            )
        else:
            print(
                f"Sweep {sweep:4d} ({sweep_time:5.1f}s, total {elapsed:6.1f}s): "
                f"log_joints=[{', '.join(f'{float(s):.0f}' for s in scores)}]"
            )

        # Checkpoint
        if sweep % CKPT_EVERY == 0:
            best_idx = int(jnp.argmax(scores))
            save_checkpoint(all_packed[best_idx], str(CKPT_DIR), sweep, column_types=column_types)
            print(f"  Checkpoint saved: sweep_{sweep}")

        gc.collect()

    total_time = time.time() - t_total
    print(f"\nCompleted {N_SWEEPS - start_sweep} sweeps in {total_time:.1f}s")
    print(f"({total_time / max(1, N_SWEEPS - start_sweep):.2f}s per sweep, {n_devices} GPUs)")

# Select best chain and prepare for queries
all_scores = jnp.array([float(packed_log_joint(p, data_jax)) for p in all_packed])
best_idx = int(jnp.argmax(all_scores))
best_packed = all_packed[best_idx]
all_chains = all_packed  # Keep all for multi-chain queries
print(f"Best chain: {best_idx} (log-joint: {float(all_scores[best_idx]):.0f}")

## 7. Convergence Diagnostics

**Note on Rhat for structure-learning models:** The Gelman-Rubin Rhat statistic measures inter-chain agreement and is designed for unimodal targets. CrossCat's partition space is combinatorial — with 20 columns, the number of possible view partitions is the Bell number B(20) ~ 5 x 10^13. Different chains legitimately settle into different posterior modes (structural hypotheses). A high Rhat does **not** mean individual chains haven't converged — it means chains found different structures. The key diagnostic is that **per-chain log-joint traces stabilize** (which they do by ~300 sweeps). We select the best chain (highest log-joint) for row-level queries and average over all chains for dependence structure.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Log-joint traces per chain
sweep_indices = [(i + 1) * DIAG_EVERY for i in range(len(log_joint_traces[0]))]
for c in range(N_CHAINS):
    ax1.plot(sweep_indices, log_joint_traces[c], marker="o", markersize=3, label=f"Chain {c}")
ax1.set_xlabel("Sweep")
ax1.set_ylabel("Log-Joint")
ax1.set_title("Per-Chain Log-Joint Convergence")
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# Rhat over time
rhat_values = []
for t in range(4, len(log_joint_traces[0]) + 1):
    traces_t = jnp.array([tr[:t] for tr in log_joint_traces])
    rhat_values.append(float(gelman_rubin_rhat(traces_t)))
rhat_sweeps = sweep_indices[3:]
ax2.plot(rhat_sweeps, rhat_values, "b-o", markersize=4)
ax2.axhline(y=1.1, color="red", linestyle="--", label="Rhat = 1.1 threshold")
ax2.axhline(y=1.0, color="green", linestyle=":", alpha=0.5)
ax2.set_xlabel("Sweep")
ax2.set_ylabel("Gelman-Rubin Rhat")
ax2.set_title("Convergence Diagnostic")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(CACHE_DIR / "convergence.png", dpi=150, bbox_inches="tight")
plt.show()
display(Image(str(CACHE_DIR / "convergence.png")))

final_rhat = rhat_values[-1] if rhat_values else float("nan")
print(f"Final Rhat: {final_rhat:.3f}")
if final_rhat < 1.1:
    print("Chains converged (Rhat < 1.1)")
else:
    print("WARNING: Rhat > 1.1 — consider running more sweeps")

## 8. Dependence Structure Discovery (Z-Matrix)

The Z-matrix shows pairwise dependence probabilities between material properties.
High values mean two properties almost always end up in the same view (statistically dependent).
Low values mean they're in different views (independent).

This is the flagship result: supervised ML predicts one property at a time, but CrossCat
reveals the **joint** dependency structure across all properties simultaneously.

In [ ]:
# Compute Z-matrix across all chains
z_matrix = np.array(packed_dependence_matrix(all_chains))
print(f"Z-matrix shape: {z_matrix.shape}")

# Reorder columns by hierarchical clustering for cleaner visualization
from scipy.cluster.hierarchy import leaves_list, linkage
from scipy.spatial.distance import squareform

dist_matrix = 1.0 - z_matrix
np.fill_diagonal(dist_matrix, 0)
dist_matrix = (dist_matrix + dist_matrix.T) / 2
dist_matrix = np.maximum(dist_matrix, 0)
condensed = squareform(dist_matrix)
linkage_matrix = linkage(condensed, method="average")
order = leaves_list(linkage_matrix)

z_ordered = z_matrix[np.ix_(order, order)]
ordered_names = [col_names[i] for i in order]

# Plot
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Original order (grouped by domain)
im0 = axes[0].imshow(z_matrix, cmap="YlOrRd", vmin=0, vmax=1)
axes[0].set_xticks(range(n_cols))
axes[0].set_yticks(range(n_cols))
axes[0].set_xticklabels(col_names, rotation=45, ha="right", fontsize=7)
axes[0].set_yticklabels(col_names, fontsize=7)
axes[0].set_title("Z-Matrix (domain order)")
plt.colorbar(im0, ax=axes[0], shrink=0.8, label="P(same view)")

# Clustered order
im1 = axes[1].imshow(z_ordered, cmap="YlOrRd", vmin=0, vmax=1)
axes[1].set_xticks(range(n_cols))
axes[1].set_yticks(range(n_cols))
axes[1].set_xticklabels(ordered_names, rotation=45, ha="right", fontsize=7)
axes[1].set_yticklabels(ordered_names, fontsize=7)
axes[1].set_title("Z-Matrix (hierarchically clustered)")
plt.colorbar(im1, ax=axes[1], shrink=0.8, label="P(same view)")

plt.tight_layout()
plt.savefig(CACHE_DIR / "z_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
display(Image(str(CACHE_DIR / "z_matrix.png")))

# Print top dependencies
pairs = []
for i in range(n_cols):
    for j in range(i + 1, n_cols):
        pairs.append((col_names[i], col_names[j], float(z_matrix[i, j])))
pairs.sort(key=lambda x: -x[2])

print("\nTop 15 property dependencies:")
print(f"{'Property A':30s} {'Property B':30s} {'Dependence':>10}")
print("-" * 72)
for a, b, score in pairs[:15]:
    print(f"{a:30s} {b:30s} {score:>10.3f}")

In [ ]:
# Show the discovered view structure: which properties cluster together

best_state = unpack_state(best_packed, column_types, data=data_jax)

print(f"Discovered {len(best_state.views)} views (independent property groups):")
print("=" * 70)
for i, view in enumerate(best_state.views):
    view_cols = [col_names[j] for j in view.column_indices]
    n_clusters = len(set(int(a) for a in view.row_assignments))
    cluster_counts = Counter(int(a) for a in view.row_assignments)
    sizes = sorted(cluster_counts.values(), reverse=True)
    print(f"\n  View {i} ({n_clusters} material clusters):")
    print(f"    Properties: {view_cols}")
    print(f"    Cluster sizes: {sizes[:10]}{'...' if len(sizes) > 10 else ''}")

In [ ]:
# Physics validation: check expected property groupings


def check_grouping(name_a, name_b, z_mat, names):
    """Check if two properties are in the same view."""
    if name_a in names and name_b in names:
        i, j = names.index(name_a), names.index(name_b)
        dep = float(z_mat[i, j])
        status = "GROUPED" if dep > 0.5 else "SEPARATED"
        return f"  {name_a} <-> {name_b}: {dep:.3f} ({status})"
    return None


print("Physics validation — expected groupings:")
expected = [
    ("Band Gap (eV)", "Is Metal", "electronic"),
    ("Band Gap (eV)", "Electronic Dielectric", "electronic"),
    ("Bulk Modulus (GPa)", "Shear Modulus (GPa)", "mechanical"),
    ("Formation Energy (eV/atom)", "Is Stable", "thermodynamic"),
    ("Formation Energy (eV/atom)", "E Above Hull (eV/atom)", "thermodynamic"),
    ("Density (g/cm3)", "Volume (A3)", "structural"),
    ("Is Metal", "Electronic Dielectric", "Penn model"),
    ("N Elements", "Formation Energy (eV/atom)", "complexity"),
    # v2: compositional + ordinal pairs
    ("Avg Electronegativity", "Band Gap (eV)", "electronic"),
    ("Avg Ionic Radius (A)", "Density (g/cm3)", "structural"),
    ("Laue Class", "Crystal System", "symmetry"),
]

for name_a, name_b, domain in expected:
    result = check_grouping(name_a, name_b, z_matrix, col_names)
    if result:
        print(f"  [{domain:14s}] {result}")

## 9. Anomaly Detection — Unusual Materials

Row typicality scores identify materials with unusual property combinations.
These are candidates for further experimental investigation — they don't fit
any discovered cluster well.

In [ ]:
# Compute row typicality for all materials
print("Computing material typicality scores...")
t0 = time.time()

typicality_scores = np.array(batch_row_typicality([best_packed], jnp.arange(n_rows)))

print(f"Computed in {time.time() - t0:.1f}s")


# Create anomaly DataFrame
anomaly_df = pd.DataFrame(
    {
        "Material ID": material_ids,
        "Formula": formulas,
        "Crystal System": [
            CRYSTAL_SYSTEM_MAP
            and {v: k for k, v in CRYSTAL_SYSTEM_MAP.items()}.get(
                int(data_np[i, valid_attrs.index("crystal_system")]), "?"
            )
            if "crystal_system" in valid_attrs
            and not np.isnan(data_np[i, valid_attrs.index("crystal_system")])
            else "?"
            for i in range(n_rows)
        ],
        "Typicality": typicality_scores,
    }
).sort_values("Typicality")

print("\nMost ATYPICAL materials (lowest typicality = most unusual):")
print("=" * 70)
for _, row in anomaly_df.head(20).iterrows():
    print(
        f"  {row['Typicality']:.3f}  {row['Formula']:20s}  {row['Crystal System']:15s}  {row['Material ID']}"
    )

print("\nMost TYPICAL materials (highest typicality):")
for _, row in anomaly_df.tail(5).iterrows():
    print(
        f"  {row['Typicality']:.3f}  {row['Formula']:20s}  {row['Crystal System']:15s}  {row['Material ID']}"
    )

In [ ]:
# Visualize typicality distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram
axes[0].hist(typicality_scores, bins=50, edgecolor="black", alpha=0.7, color="steelblue")
axes[0].axvline(
    x=np.percentile(typicality_scores, 1), color="red", linestyle="--", label="1st percentile"
)
axes[0].set_xlabel("Typicality Score")
axes[0].set_ylabel("Count")
axes[0].set_title("Material Typicality Distribution")
axes[0].legend()

# Scatter: typicality vs formation energy
fe_idx = (
    col_names.index("Formation Energy (eV/atom)")
    if "Formation Energy (eV/atom)" in col_names
    else None
)
if fe_idx is not None:
    fe_vals = data_np[:, fe_idx]
    valid_mask = ~np.isnan(fe_vals)
    scatter = axes[1].scatter(
        fe_vals[valid_mask],
        typicality_scores[valid_mask],
        c=typicality_scores[valid_mask],
        cmap="RdYlBu",
        s=5,
        alpha=0.5,
    )
    plt.colorbar(scatter, ax=axes[1], label="Typicality")
    axes[1].set_xlabel("Formation Energy (eV/atom)")
    axes[1].set_ylabel("Typicality Score")
    axes[1].set_title("Typicality vs Formation Energy")

    # Annotate top anomalies
    top5 = anomaly_df.head(5)
    for _, row in top5.iterrows():
        idx = material_ids.index(row["Material ID"])
        if valid_mask[idx]:
            axes[1].annotate(
                row["Formula"], (fe_vals[idx], typicality_scores[idx]), fontsize=7, ha="left"
            )

plt.tight_layout()
plt.savefig(CACHE_DIR / "anomaly_detection.png", dpi=150, bbox_inches="tight")
plt.show()
display(Image(str(CACHE_DIR / "anomaly_detection.png")))

In [ ]:
# Cell-level anomaly: which properties are surprising for the top anomalous materials?

top_anomalous = anomaly_df.head(5)
print("Cell-level anomaly attribution for top-5 unusual materials:")
print("=" * 70)

for _, row in top_anomalous.iterrows():
    idx = material_ids.index(row["Material ID"])
    print(f"\n{row['Formula']} ({row['Material ID']}, typicality={row['Typicality']:.3f}):")

    col_scores = []
    for col_j in range(n_cols):
        val = data_jax[idx, col_j]
        if jnp.isnan(val):
            continue
        log_p = packed_predictive_probability(
            best_packed,
            data_jax,
            query_cols=jnp.array([col_j]),
            query_vals=jnp.array([float(val)]),
            row_id=idx,
        )
        col_scores.append((col_names[col_j], float(val), float(log_p)))

    col_scores.sort(key=lambda x: x[2])
    for name, val, lp in col_scores[:5]:
        print(f"    {name:30s}  value={val:8.3f}  log_p={lp:.2f}")

## 10. Missing Property Imputation

The headline practical result: predict missing mechanical properties (bulk modulus,
shear modulus) from electronic and structural data. DFT elasticity calculations are
expensive — CrossCat can fill gaps using the discovered dependency structure.

We evaluate quality with a 10% holdout of observed values.

**Caveats on per-column R²:**
- R² for **categorical columns** (Crystal System, Magnetic Ordering) is not meaningful — these are discrete classifications, not regressions. Use accuracy or F1 instead.
- **Poisson Ratio** has a very narrow physical range (~0.1–0.4 for most materials), so even small absolute errors produce large negative R². The MAE of 0.08 is actually reasonable for screening.
- Columns with **low holdout counts** (e.g., Bulk/Shear Modulus with ~170–200 samples) have higher variance in R² estimates.
- The **overall R²** is dominated by high-variance columns; the **median per-column R²** is a more robust summary.

In [ ]:
# Hold-out evaluation: mask 10% of observed values and measure recovery

rng_imp = np.random.default_rng(SEED + 200)
holdout_frac = 0.10

# Create holdout mask (only mask cells that actually have data)
observed_mask = ~np.isnan(data_np)
n_observed = observed_mask.sum()
n_holdout = int(n_observed * holdout_frac)

observed_indices = np.argwhere(observed_mask)
holdout_idx = rng_imp.choice(len(observed_indices), size=n_holdout, replace=False)
holdout_mask = np.zeros_like(observed_mask)
for idx in holdout_idx:
    r, c = observed_indices[idx]
    holdout_mask[r, c] = True

print(f"Holdout: {n_holdout} cells ({holdout_frac:.0%} of {n_observed} observed)")

# Create masked data
data_masked_np = data_np.copy()
data_masked_np[holdout_mask] = np.nan
data_masked = jnp.array(data_masked_np)

# Impute per column and collect metrics
imp_key = jax.random.key(SEED + 300)
per_col_metrics = {}

print("\nImputation quality (10% holdout):")
print(f"{'Column':30s} {'N Holdout':>10} {'MAE':>10} {'RMSE':>10} {'R2':>10}")
print("-" * 72)

all_true = []
all_pred = []

for col_j in range(n_cols):
    col_holdout_rows = np.where(holdout_mask[:, col_j])[0]
    if len(col_holdout_rows) == 0:
        continue

    imp_key, subkey = jax.random.split(imp_key)
    values, confidences = batch_impute_column(
        subkey,
        best_packed,
        data_masked,
        query_col=col_j,
        row_ids=jnp.array(col_holdout_rows),
    )

    true_vals = data_np[col_holdout_rows, col_j]
    pred_vals = np.array(values)

    # Filter valid predictions
    valid = ~np.isnan(pred_vals) & ~np.isnan(true_vals)
    if valid.sum() < 2:
        continue

    true_v = true_vals[valid]
    pred_v = pred_vals[valid]

    mae = np.mean(np.abs(true_v - pred_v))
    rmse = np.sqrt(np.mean((true_v - pred_v) ** 2))
    ss_res = np.sum((true_v - pred_v) ** 2)
    ss_tot = np.sum((true_v - true_v.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")

    per_col_metrics[col_names[col_j]] = {"mae": mae, "rmse": rmse, "r2": r2, "n": int(valid.sum())}
    print(f"{col_names[col_j]:30s} {int(valid.sum()):>10} {mae:>10.4f} {rmse:>10.4f} {r2:>10.3f}")

    all_true.extend(true_v.tolist())
    all_pred.extend(pred_v.tolist())

# Overall metrics
all_true = np.array(all_true)
all_pred = np.array(all_pred)
overall_mae = np.mean(np.abs(all_true - all_pred))
overall_r2 = 1 - np.sum((all_true - all_pred) ** 2) / np.sum((all_true - all_true.mean()) ** 2)
print(f"\nOverall: MAE={overall_mae:.4f}, R2={overall_r2:.3f}")

In [ ]:
# Parity plots for key columns (true vs predicted)

# Pick columns with most holdout data
plot_cols = sorted(per_col_metrics.keys(), key=lambda k: -per_col_metrics[k]["n"])[:4]

fig, axes = plt.subplots(1, len(plot_cols), figsize=(5 * len(plot_cols), 5))
if len(plot_cols) == 1:
    axes = [axes]

for ax, col_name in zip(axes, plot_cols):
    col_j = col_names.index(col_name)
    col_holdout_rows = np.where(holdout_mask[:, col_j])[0]

    imp_key, subkey = jax.random.split(imp_key)
    values, _ = batch_impute_column(
        subkey,
        best_packed,
        data_masked,
        query_col=col_j,
        row_ids=jnp.array(col_holdout_rows),
    )

    true_v = data_np[col_holdout_rows, col_j]
    pred_v = np.array(values)
    valid = ~np.isnan(pred_v) & ~np.isnan(true_v)

    ax.scatter(true_v[valid], pred_v[valid], s=5, alpha=0.3, color="steelblue")
    lims = [
        min(true_v[valid].min(), pred_v[valid].min()),
        max(true_v[valid].max(), pred_v[valid].max()),
    ]
    ax.plot(lims, lims, "r--", alpha=0.7, label="Perfect")
    ax.set_xlabel("True")
    ax.set_ylabel("Predicted")
    m = per_col_metrics[col_name]
    ax.set_title(f"{col_name}\nR2={m['r2']:.3f}, MAE={m['mae']:.4f}")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(CACHE_DIR / "imputation_parity.png", dpi=150, bbox_inches="tight")
plt.show()
display(Image(str(CACHE_DIR / "imputation_parity.png")))

In [ ]:
# Impute actually-missing elasticity values and show distribution

elastic_cols = [
    ("Bulk Modulus (GPa)", "bulk_modulus_vrh"),
    ("Shear Modulus (GPa)", "shear_modulus_vrh"),
]

fig, axes = plt.subplots(1, len(elastic_cols), figsize=(7 * len(elastic_cols), 5))
if len(elastic_cols) == 1:
    axes = [axes]

for ax, (disp_name, attr_name) in zip(axes, elastic_cols):
    if disp_name not in col_names:
        continue
    col_j = col_names.index(disp_name)

    observed_vals = data_np[~np.isnan(data_np[:, col_j]), col_j]
    missing_rows = np.where(np.isnan(data_np[:, col_j]))[0]

    if len(missing_rows) == 0:
        ax.text(0.5, 0.5, "No missing values", transform=ax.transAxes, ha="center")
        continue

    imp_key, subkey = jax.random.split(imp_key)
    imputed_vals, confidences = batch_impute_column(
        subkey,
        best_packed,
        data_jax,
        query_col=col_j,
        row_ids=jnp.array(missing_rows),
    )
    imputed_vals = np.array(imputed_vals)
    confidences = np.array(confidences)

    ax.hist(
        observed_vals,
        bins=40,
        alpha=0.6,
        label=f"Observed (n={len(observed_vals)})",
        color="steelblue",
        density=True,
    )
    ax.hist(
        imputed_vals[~np.isnan(imputed_vals)],
        bins=40,
        alpha=0.6,
        label=f"Imputed (n={len(missing_rows)})",
        color="coral",
        density=True,
    )
    ax.set_xlabel(disp_name)
    ax.set_ylabel("Density")
    ax.set_title(f"{disp_name}: Observed vs Imputed")
    ax.legend()

    mean_conf = np.nanmean(confidences)
    print(f"{disp_name}: imputed {len(missing_rows)} values, mean confidence={mean_conf:.3f}")

plt.tight_layout()
plt.savefig(CACHE_DIR / "elasticity_imputation.png", dpi=150, bbox_inches="tight")
plt.show()
display(Image(str(CACHE_DIR / "elasticity_imputation.png")))

## 11. Mutual Information — Property Relationships

Mutual information quantifies how much knowing one property tells you about another.
Unlike Pearson correlation, MI captures **nonlinear** relationships.

In [ ]:
# Compute MI for domain-relevant pairs

INTERESTING_PAIRS = [
    ("Band Gap (eV)", "Total Dielectric"),
    ("Band Gap (eV)", "Is Metal"),
    ("Band Gap (eV)", "Electronic Dielectric"),
    ("Formation Energy (eV/atom)", "Is Stable"),
    ("Formation Energy (eV/atom)", "E Above Hull (eV/atom)"),
    ("Bulk Modulus (GPa)", "Shear Modulus (GPa)"),
    ("Bulk Modulus (GPa)", "Density (g/cm3)"),
    ("Density (g/cm3)", "Volume (A3)"),
    ("Is Metal", "Electronic Dielectric"),
    ("Crystal System", "Elastic Anisotropy"),
    ("N Elements", "Formation Energy (eV/atom)"),
    ("E Above Hull (eV/atom)", "Is Stable"),
    # v2: compositional + ordinal pairs
    ("Avg Electronegativity", "Band Gap (eV)"),
    ("Avg Ionic Radius (A)", "Density (g/cm3)"),
    ("Laue Class", "Crystal System"),
]

mi_key = jax.random.key(SEED + 400)
mi_results = []

print("Computing mutual information for property pairs...")
for name_a, name_b in INTERESTING_PAIRS:
    if name_a not in col_names or name_b not in col_names:
        continue
    col_i = col_names.index(name_a)
    col_j = col_names.index(name_b)

    mi_key, subkey = jax.random.split(mi_key)
    mi_val, linfoot_val = packed_mutual_information(
        all_chains,
        column_types,
        col_i=col_i,
        col_j=col_j,
        rng_key=subkey,
    )
    mi_val = float(mi_val)
    linfoot = float(linfoot_val)

    mi_results.append(
        {
            "Property A": name_a,
            "Property B": name_b,
            "MI": mi_val,
            "Linfoot": linfoot,
        }
    )
    print(f"  {name_a:30s} <-> {name_b:30s}  MI={mi_val:.3f}  Linfoot={linfoot:.3f}")

mi_df = pd.DataFrame(mi_results).sort_values("Linfoot", ascending=False)
mi_df

In [ ]:
# Visualize MI results
fig, ax = plt.subplots(figsize=(12, max(6, len(mi_df) * 0.4)))

pair_labels = [f"{r['Property A']} <-> {r['Property B']}" for _, r in mi_df.iterrows()]
linfoot_vals = mi_df["Linfoot"].values

colors = plt.cm.YlOrRd(linfoot_vals / max(linfoot_vals.max(), 0.01))
bars = ax.barh(range(len(mi_df)), linfoot_vals, color=colors)
ax.set_yticks(range(len(mi_df)))
ax.set_yticklabels(pair_labels, fontsize=9)
ax.set_xlabel("Linfoot Correlation (normalized MI)")
ax.set_title("Mutual Information Between Material Properties")
ax.invert_yaxis()

# Annotate values
for i, (_, r) in enumerate(mi_df.iterrows()):
    ax.text(r["Linfoot"] + 0.01, i, f"{r['Linfoot']:.3f}", va="center", fontsize=8)

plt.tight_layout()
plt.savefig(CACHE_DIR / "mutual_information.png", dpi=150, bbox_inches="tight")
plt.show()
display(Image(str(CACHE_DIR / "mutual_information.png")))

## 12. Generative Classification — Predicting Metallicity

CrossCat can classify without ever training a classifier: it uses the posterior predictive
P(is_metal | other properties) from the generative model. This is Bayesian model averaging
over all discovered structures.

**Important context:** Only 3.4% of materials in this dielectric-selected dataset are metals
(250 / 7,327). This extreme class imbalance means standard accuracy is misleading — a naive
"predict all non-metal" baseline achieves 96.6% accuracy. Instead, we evaluate the model's
**ranking ability**: does P(metal) correlate with actual metallicity? We optimize the
classification threshold for F1 rather than using the default 0.5 cutoff, and examine the
probability distributions for true metals vs. non-metals. The model's value here is as a
**metallicity ranker** — surfacing metal-like materials from their property profiles — rather
than a binary classifier.

In [ ]:
# Classify is_metal — analyze posterior probabilities, not just argmax

if "Is Metal" in col_names:
    metal_col = col_names.index("Is Metal")

    actual_labels = data_np[:, metal_col]
    valid_rows = ~np.isnan(actual_labels)
    valid_row_ids = np.where(valid_rows)[0]

    print(f"Classifying is_metal for {len(valid_row_ids)} materials...")
    t0 = time.time()

    log_probs = batch_classify_column(
        best_packed,
        data_jax,
        target_col=metal_col,
        candidate_vals=jnp.array([0.0, 1.0]),
        row_ids=jnp.array(valid_row_ids),
    )

    # Convert to probabilities
    probs = np.array(jax.nn.softmax(log_probs, axis=1))
    p_metal = probs[:, 1]  # P(is_metal=1)

    print(f"Classified in {time.time() - t0:.1f}s")

    true_labels = actual_labels[valid_rows]
    n_metal = int(true_labels.sum())
    n_nonmetal = len(true_labels) - n_metal
    print(
        f"Class balance: {n_metal} metal / {n_nonmetal} non-metal ({100 * n_metal / len(true_labels):.1f}% metal)"
    )

    # Show probability distribution for actual metals vs non-metals
    p_metal_for_metals = p_metal[true_labels == 1]
    p_metal_for_nonmetals = p_metal[true_labels == 0]

    print(
        f"\nP(metal) for actual metals:     mean={p_metal_for_metals.mean():.3f}, median={np.median(p_metal_for_metals):.3f}"
    )
    print(
        f"P(metal) for actual non-metals: mean={p_metal_for_nonmetals.mean():.3f}, median={np.median(p_metal_for_nonmetals):.3f}"
    )

    # Use a threshold tuned for the imbalanced class
    # Find threshold that maximizes F1
    best_f1 = 0
    best_thresh = 0.5
    for thresh in np.arange(0.01, 0.50, 0.01):
        preds = (p_metal >= thresh).astype(float)
        tp = ((preds == 1) & (true_labels == 1)).sum()
        fp = ((preds == 1) & (true_labels == 0)).sum()
        fn = ((preds == 0) & (true_labels == 1)).sum()
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh

    # Apply best threshold
    predictions = (p_metal >= best_thresh).astype(float)
    tp = int(((predictions == 1) & (true_labels == 1)).sum())
    tn = int(((predictions == 0) & (true_labels == 0)).sum())
    fp = int(((predictions == 1) & (true_labels == 0)).sum())
    fn = int(((predictions == 0) & (true_labels == 1)).sum())
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    accuracy = (tp + tn) / len(true_labels)

    print(f"\nOptimal threshold: {best_thresh:.2f}")
    print(f"  Accuracy:  {accuracy:.3f}")
    print(f"  Precision: {precision:.3f}")
    print(f"  Recall:    {recall:.3f}")
    print(f"  F1 Score:  {f1:.3f}")
    print(f"  TP={tp} FP={fp} FN={fn} TN={tn}")

    # Top materials ranked by P(metal) — are the actual metals near the top?
    rank_df = pd.DataFrame(
        {
            "Formula": [formulas[i] for i in valid_row_ids],
            "Actual": true_labels,
            "P(metal)": p_metal,
        }
    ).sort_values("P(metal)", ascending=False)

    print("\nTop 20 by P(metal):")
    for _, row in rank_df.head(20).iterrows():
        marker = " <<<" if row["Actual"] == 1 else ""
        print(f"  P={row['P(metal)']:.3f}  actual={int(row['Actual'])}  {row['Formula']}{marker}")
else:
    print("Is Metal column not available")

In [ ]:
# Confusion matrix visualization

if "Is Metal" in col_names:
    cm = np.array([[tn, fp], [fn, tp]])

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Non-Metal", "Metal"])
    ax.set_yticklabels(["Non-Metal", "Metal"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(f"Metallicity Classification\nAccuracy={accuracy:.3f}, F1={f1:.3f}")

    for i in range(2):
        for j in range(2):
            ax.text(
                j,
                i,
                f"{cm[i, j]}",
                ha="center",
                va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black",
                fontsize=16,
            )

    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(CACHE_DIR / "classification.png", dpi=150, bbox_inches="tight")
    plt.show()
    display(Image(str(CACHE_DIR / "classification.png")))

## 13. Summary

In [ ]:
# Collect all results
summary = {
    "config": {
        "n_materials": n_rows,
        "n_columns": n_cols,
        "n_chains": N_CHAINS,
        "n_sweeps": N_SWEEPS,
        "max_clusters": MAX_CLUSTERS,
        "max_views": MAX_VIEWS,
        "seed": SEED,
    },
    "convergence": {
        "final_rhat": final_rhat,
        "converged": final_rhat < 1.1,
        "total_time_s": total_time,
    },
    "structure": {
        "n_views": len(best_state.views),
        "top_dependencies": [(a, b, s) for a, b, s in pairs[:10]],
    },
    "imputation": {
        "overall_mae": float(overall_mae),
        "overall_r2": float(overall_r2),
        "per_column": {
            k: {kk: float(vv) for kk, vv in v.items()} for k, v in per_col_metrics.items()
        },
    },
    "classification": {
        "accuracy": float(accuracy) if "Is Metal" in col_names else None,
        "f1": float(f1) if "Is Metal" in col_names else None,
    },
}

# Save summary
with open(CACHE_DIR / "summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print("Results Summary")
print("=" * 50)
print(f"Materials: {n_rows}, Properties: {n_cols}")
print(f"Views discovered: {len(best_state.views)}")
print(f"Convergence: Rhat={final_rhat:.3f}")
print(f"Imputation: MAE={overall_mae:.4f}, R2={overall_r2:.3f}")
if "Is Metal" in col_names:
    print(f"Classification: Accuracy={accuracy:.3f}, F1={f1:.3f}")
print(f"\nResults saved to: {CACHE_DIR}")

In [ ]:
# Save final model
model_path = CACHE_DIR / "best_model.jxc"
save_packed_state(best_packed, str(model_path), column_types=column_types)
print(f"Model saved to: {model_path}")
print("\nTo reload in a future session:")
print("  from crosscat import load_packed_state")
print(f"  packed, col_types = load_packed_state('{model_path}')")

In [ ]:
# === Download results ===
# On Kaggle: results are saved under examples/results/materials_project/
# Use Kaggle's "Save & Run All" then download from Output tab.
#
# Files:
#   - mp_dielectric_cache.parquet  (raw data cache)
#   - best_model.jxc              (trained CrossCat model)
#   - summary.json                (all metrics)
#   - missingness.png             (data coverage visualization)
#   - z_matrix.png                (dependence structure)
#   - convergence.png             (chain convergence)
#   - anomaly_detection.png       (typicality distribution)
#   - imputation_parity.png       (true vs predicted)
#   - elasticity_imputation.png   (observed vs imputed distributions)
#   - mutual_information.png      (MI bar chart)
#   - classification.png          (confusion matrix)
#   - crystal_systems.png         (data composition)

import os

print("Output files:")
for f in sorted(CACHE_DIR.iterdir()):
    size = f.stat().st_size
    if size > 1024 * 1024:
        print(f"  {f.name:40s}  {size / 1024 / 1024:.1f} MB")
    else:
        print(f"  {f.name:40s}  {size / 1024:.1f} KB")